# extract_e07
Phase 2 Stage 2 Branch C L4 GPU extraction for E07 unique genuinely-changed SCREEN `image_caption` views (both `last_caption` and `caption_mean` on the same midpoint forward: `qwen` depth 18, `gemma` depth 17). Run cells 2, 3, and 4 in order. Configure only cell 3. Use a Colab L4 GPU runtime with `requirements-l4.txt`.

In [ ]:
import json, os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned L4 requirements.')
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/Research-phase2')
CODE_REF = 'phase2'  # Set to the approved Phase 2 branch or commit SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-l4.txt')], check=True)
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Colab L4 CUDA runtime with bfloat16 support is required.')
gpu_name = torch.cuda.get_device_name(0)
if 'L4' not in gpu_name:
    raise RuntimeError(f'Expected NVIDIA L4 GPU; got {gpu_name!r}')
print('Fixed code commit:', CODE_COMMIT)
print('GPU:', gpu_name)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage 2 E07 extraction settings (edit only this cell).
MODEL = 'qwen'  # 'qwen' (depth 18) or 'gemma' (depth 17; requires HF_TOKEN in Colab Secrets).
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
PLAN_REF = {
    'path': 'PASTE_PHASE2_PLAN_COMPLETE_JSON_PATH',
    'sha256': 'PASTE_PHASE2_PLAN_SHA256',
}
E07_AUDIT_REF = {
    'path': 'PASTE_PHASE2_E07_AUDIT_COMPLETE_JSON_PATH',
    'sha256': 'PASTE_PHASE2_E07_AUDIT_SHA256',
}
# Optional explicit chunk resume map ({0: {'path': '...', 'sha256': '...'}}) or None.
RESUME_REFS = None


In [ ]:
import re
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError(f'Invalid MODEL selector: {MODEL!r}')
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 2 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Mounted PHASE1_ROOT not found: {PHASE1_ROOT}')
for label, ref in (('PLAN_REF', PLAN_REF), ('E07_AUDIT_REF', E07_AUDIT_REF)):
    manifest = PHASE1_ROOT / ref['path']
    if not manifest.is_file():
        raise RuntimeError(f'Missing mounted {label} manifest: {manifest}')
run_env = os.environ.copy()
if MODEL == 'gemma':
    hf_token = run_env.get('HF_TOKEN')
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError('Gemma requires HF_TOKEN in Colab Secrets (userdata) or environment.')
    run_env['HF_TOKEN'] = hf_token
def redact(text):
    t = run_env.get('HF_TOKEN')
    if t:
        text = text.replace(t, '[REDACTED]')
    return re.sub(r'hf_[A-Za-z0-9]+', '[REDACTED]', text)
command = [
    sys.executable, '-u', '-m', 'src.phase2.extract_e07',
    '--model', MODEL,
    '--phase1-root', str(PHASE1_ROOT),
    '--plan', json.dumps(PLAN_REF),
    '--e07-audit', json.dumps(E07_AUDIT_REF),
    '--storage-scope', 'persistent-drive',
]
if RESUME_REFS is not None:
    command.extend(['--resume-refs', json.dumps(RESUME_REFS)])
process = subprocess.Popen(
    command, cwd=REPO, env=run_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
blockers = []
for line in process.stdout:
    safe_line = redact(line)
    print(safe_line, end='', flush=True)
    try:
        payload = json.loads(safe_line)
        if isinstance(payload, dict) and payload.get('status') == 'blocked':
            blockers.extend(payload.get('blockers', []))
    except json.JSONDecodeError:
        pass
code = process.wait()
if code != 0:
    detail = '; '.join(redact(str(b)) for b in blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'E07 GPU extraction failed ({code}): {detail}')
